In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.evt import fit_pot, evt_threshold
from src.evaluation import tpr_at_threshold

SC = ROOT / "results" / "scores"
files = {
    "M1 IsolationForest": "m1_iforest_nftoniot_seed11.parquet",
    "M2 OneClassSVM": "m2_ocsvm_nftoniot_seed11.parquet",
    "M3 AE v1": "m3_autoencoder_nftoniot_seed11.parquet",
    "M3 AE v2": "m3v2_autoencoder_nftoniot_seed11.parquet",
}
QS = [1e-2, 1e-3, 1e-4]

data, diag = {}, []
for name, f in files.items():
    df = pd.read_parquet(SC / f)
    data[name] = df
    val = df[(df["split"] == "val") & (df["label"] == 0)]["score"].to_numpy()
    fit = fit_pot(val)
    data[name + "_fit"] = (fit, val)
    diag.append({"model": name, **{k: round(v, 4) for k, v in fit.items()}})
pd.DataFrame(diag)

,model,t,xi,sigma,n,n_t,ks_stat,ks_p
0,M1 IsolationForest,0.7120,-0.3699,0.0411,539110,26860,0.1666,0.0
1,M2 OneClassSVM,12.2563,-1.0675,43.7611,539110,26956,0.0739,0.0
2,M3 AE v1,0.1659,0.4395,0.3274,539110,26956,0.0809,0.0
3,M3 AE v2,0.0057,0.6629,0.0147,539110,26258,0.0473,0.0


In [2]:
rows = []
for name in files:
    df = data[name]
    fit, val = data[name + "_fit"]
    for q in QS:
        for method, thr in [("EVT", evt_threshold(fit, q)), ("percentile", float(np.quantile(val, 1 - q)))]:
            for sp in ["tune", "test"]:
                d = df[df["split"] == sp]
                if d.empty:
                    continue
                tpr, fpr = tpr_at_threshold(d, thr)
                macro = np.mean([(g["score"] > thr).mean()
                                 for _, g in d[d["label"] == 1].groupby("attack_type")])
                rows.append({"model": name, "target_fpr": q, "method": method, "split": sp,
                             "real_fpr": fpr, "tpr": tpr, "macro_tpr": macro})

evt_table = pd.DataFrame(rows).round(4)
evt_table.to_csv(ROOT / "results" / "evt_comparison.csv", index=False)
evt_table[evt_table["split"] == "tune"]

,model,target_fpr,method,split,real_fpr,tpr,macro_tpr
0,M1 IsolationForest,0.0100,EVT,tune,0.0175,0.0144,0.0764
2,M1 IsolationForest,0.0100,percentile,tune,0.0108,0.0052,0.0204
4,M1 IsolationForest,0.0010,EVT,tune,0.0005,0.0003,0.0030
6,M1 IsolationForest,0.0010,percentile,tune,0.0012,0.0008,0.0066
8,M1 IsolationForest,0.0001,EVT,tune,0.0001,0.0000,0.0002
10,M1 IsolationForest,0.0001,percentile,tune,0.0002,0.0001,0.0016
12,M2 OneClassSVM,0.0100,EVT,tune,0.0138,0.0030,0.0203
14,M2 OneClassSVM,0.0100,percentile,tune,0.0154,0.0037,0.0242
16,M2 OneClassSVM,0.0010,EVT,tune,0.0012,0.0002,0.0007
18,M2 OneClassSVM,0.0010,percentile,tune,0.0011,0.0002,0.0007
